# Gold EDA 
**Purpose:** Explore cleaned silver data to find business insights for the dashobard. 
All charts are candidates for the final Marathos dashboard

data source: **'marathos.silver.cleaned_marathos_2.0'**

In [0]:
%python
import plotly.express as px
from pyspark.sql.functions import col, round as spark_round, avg, count

df = spark.table("marathos.silver.cleaned_marathos_2")

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")


## Yearly trend
How has ultra marathon participation grown over time?

In [0]:
yearly = spark.sql("""
    SELECT 
        year_of_event,
        COUNT(*) AS total_results,
        COUNT(DISTINCT event_name) AS unique_events
    FROM marathos.silver.cleaned_marathos_2
    GROUP BY year_of_event
    ORDER BY year_of_event
""").toPandas()

fig = px.line(
    yearly,
    x="year_of_event",
    y="total_results",
    title="Ultra marathon participation over time",
    labels={"year_of_event": "Year", "total_results": "Number of results"},
    markers=True
)
fig.show()


### Yearly trend insights
* Clear growth in participation from the 2000s onwards.
* Ultra marathon has gone from a niche sport to a global phenomenon.
* Dip around 2020 likely reflects COVID-19 pandemic cancellations.
* This trend chart will be a key visual in the dashboard.

## Country representation
Which countries produce the most ultra marathon finishers?

In [0]:
country_dist = spark.sql("""
    SELECT 
        athlete_country,
        COUNT(*) AS total_results,
        ROUND(AVG(athlete_average_speed), 2) AS avg_speed
    FROM marathos.silver.cleaned_marathos_2
    GROUP BY athlete_country
    ORDER BY total_results DESC
    LIMIT 15
""").toPandas().sort_values("total_results", ascending=False)

fig = px.bar(
    country_dist,
    x="athlete_country",
    y="total_results",
    title="Top 15 most represented countries",
    labels={"athlete_country": "Country", "total_results": "Number of results"},
    text="total_results",
    color="avg_speed",
    color_continuous_scale="Blues"
)
fig.show()

### Country insights
* USA dominates by a large margin — ultra running is deeply rooted in America.
* European countries (FRA, GBR, GER, SWE) are well represented.
* Color shows average speed, good and interesting to see if dominant countries also are the fastest.

## Gender distribution and speed


In [0]:
gender_stats = spark.sql("""
    SELECT 
        athlete_gender,
        COUNT(*) AS total_results,
        ROUND(AVG(athlete_average_speed), 3) AS avg_speed,
        ROUND(MIN(athlete_average_speed), 3) AS min_speed,
        ROUND(MAX(athlete_average_speed), 3) AS max_speed
    FROM marathos.silver.cleaned_marathos_2
    WHERE athlete_gender IN ('M', 'F')
    GROUP BY athlete_gender
    ORDER BY athlete_gender
""").toPandas()

gender_stats.display() if 'display' in dir(gender_stats) else print(gender_stats)

fig = px.bar(
    gender_stats,
    x="athlete_gender",
    y="avg_speed",
    title="Average speed by gender",
    labels={"athlete_gender": "Gender", "avg_speed": "Avg speed (km/h)"},
    text="avg_speed",
    color="athlete_gender",
    color_discrete_map={"M": "#1f77b4", "F": "#ff7f0e"}
)
fig.show()

### Gender insights
* Male athletes are on average faster than female athletes.
* Both genders have the same max speed (~20 km/h) showing the best female athletes match some of the best male athletes.

## Event type distribution
Are most events distance-based (km/mi) or time-based (h)?

In [0]:
event_type = spark.sql("""
    SELECT 
        event_type,
        COUNT(*) AS total_results,
        COUNT(DISTINCT event_name) AS unique_events
    FROM marathos.silver.cleaned_marathos_2
    GROUP BY event_type
""").toPandas()

fig = px.pie(
    event_type,
    names="event_type",
    values="total_results",
    title="Distance vs Time based events",
    color_discrete_map={"distance": "#1f77b4", "time": "#ff7f0e"}
)
fig.show()

### Event type insights
* Distance events (km/mi) are more common — these are the classic ultramarathons.
* Time events (h) are timed challenges
* Important to keep these separate in analysis since their results are measured differently (time vs distance).

## Most popular events

In [0]:
top_events = spark.sql("""
    SELECT 
        event_name,
        event_country,
        event_type,
        COUNT(*) AS total_finishers,
        COUNT(DISTINCT year_of_event) AS years_held,
        ROUND(AVG(athlete_average_speed), 2) AS avg_speed
    FROM marathos.silver.cleaned_marathos_2
    GROUP BY event_name, event_country, event_type
    ORDER BY total_finishers DESC
    LIMIT 10
""").toPandas().sort_values("total_finishers", ascending=True)

fig = px.bar(
    top_events,
    x="total_finishers",
    y="event_name",
    orientation="h",
    title="Top 10 most popular ultra marathon events",
    labels={"event_name": "Event", "total_finishers": "Total finishers"},
    text="total_finishers",
    color="event_type",
    color_discrete_map={"distance": "#1f77b4", "time": "#ff7f0e"}
)
fig.show()

### Popular events insights
* Mix of distance and time events in the top 10.
* Color shows event type wich is good for dashboard filtering.

## Age category analysis

In [0]:
age_stats = spark.sql("""
    SELECT 
        athlete_age_category,
        athlete_gender,
        COUNT(*) AS race_count,
        ROUND(AVG(athlete_average_speed), 3) AS avg_speed
    FROM marathos.silver.cleaned_marathos_2
    WHERE athlete_age_category IS NOT NULL
      AND athlete_gender IN ('M', 'F')
    GROUP BY athlete_age_category, athlete_gender
    ORDER BY athlete_age_category
""").toPandas()

fig = px.bar(
    age_stats,
    x="athlete_age_category",
    y="avg_speed",
    color="athlete_gender",
    barmode="group",
    title="Average speed by age category and gender",
    labels={
        "athlete_age_category": "Age category",
        "avg_speed": "Avg speed (km/h)",
        "athlete_gender": "Gender"
    },
    color_discrete_map={"M": "#1f77b4", "F": "#ff7f0e"}
)
fig.show()

### Age insights
* Speed peaks in younger age categories and declines with age as expected.
* Male athletes consistently faster across all age groups.
* Older age groups still competitive, shows ultramarthons are for all ages

## Speed distribution

In [0]:
speed_dist_spark = spark.sql("""
    SELECT 
        ROUND(athlete_average_speed, 1) AS speed_bucket,
        COUNT(*) AS count
    FROM marathos.silver.cleaned_marathos_2
    WHERE athlete_average_speed IS NOT NULL
    GROUP BY ROUND(athlete_average_speed, 1)
    ORDER BY speed_bucket
""")
speed_dist_spark.display()
speed_dist = speed_dist_spark.toPandas()

fig = px.bar(
    speed_dist,
    x="speed_bucket",
    y="count",
    title="Distribution of athlete average speed",
    labels={"speed_bucket": "Speed (km/h)", "count": "Number of athletes"}
)
fig.show()

### Speed distribution insights
* Speed follows a roughly normal distribution centered around 7-8 km/h.
* Long right tail is a small number of elite athletes run significantly faster.
* Very few athletes below 2 km/h or above 15 km/h, our 0.5-20 filter worked well.

## Most common race distances

In [0]:
distances = spark.sql("""
    SELECT 
        event_distance_length,
        event_distance_km,
        COUNT(*) AS total_results
    FROM marathos.silver.cleaned_marathos
    WHERE event_type = 'distance'
      AND event_distance_km IS NOT NULL
    GROUP BY event_distance_length, event_distance_km
    ORDER BY total_results DESC
    LIMIT 15
""").toPandas().sort_values("event_distance_km")

fig = px.bar(
    distances,
    x="event_distance_length",
    y="total_results",
    title="Most common race distances",
    labels={
        "event_distance_length": "Distance",
        "total_results": "Number of results"
    },
    text="total_results"
)
fig.show()

### Distance insights
* 50km is the most common ultra marathon distance.
* 100km and 100mi are also very popular and the iconic ultra distances.
* Good to know for dashboard filtering, users might want to filter by distance.

## Speed by event distance (top distances only)

In [0]:
speed_by_distance = spark.sql("""
    SELECT 
        event_distance_length,
        event_distance_km,
        ROUND(AVG(athlete_average_speed), 3) AS avg_speed,
        COUNT(*) AS total_results
    FROM marathos.silver.cleaned_marathos
    WHERE event_type = 'distance'
      AND event_distance_km IS NOT NULL
    GROUP BY event_distance_length, event_distance_km
    HAVING COUNT(*) > 1000
    ORDER BY event_distance_km
""").toPandas()

fig = px.scatter(
    speed_by_distance,
    x="event_distance_km",
    y="avg_speed",
    size="total_results",
    title="Average speed vs race distance",
    labels={
        "event_distance_km": "Distance (km)",
        "avg_speed": "Avg speed (km/h)",
        "total_results": "Number of results"
    },
    hover_data=["event_distance_length"]
)
fig.show()

### Speed vs distance insights
* As distance increases, average speed decreases as expected.
* Shorter ultras (50km) are run faster than longer ones (100mi).
* Bubble size shows popularity around 50km and 100km are the biggest bubbles.
* This is an interesting insight for the dashboard.

## Summary - Dashoboard candidates 

Based on this EDA, these are the best chrtas for the Marathos dashboard: 

Chart type insight 
- participation over time as a linechart that describes a growth trend and the covid pause
- Top 15 countries as a barchart with geographic distrubution 
- Gender speed comparison as a barchart M vs F performance 
- Event type split as a pie chart Distance vs time
- Top 10 events as a horizontal bar to describe the most popular races